# 🏋️ RATISS-ONE — L'ENTRAÎNEMENT LOURD (1h, Colab gratuit)
10 millions d'enseignements : 5M du fichier lourd + 5M fraîches générées sur place (seed 8), une nuit par lot, RAM stable (~13 Mo mesurés). Le tissu ne stocke JAMAIS la masse : chaque ligne est entendue puis oubliée — seules les forces restent.

**Vérité GPU** : notre tissu est CPU pur (entiers + dictionnaires) — le GPU ne l'accélère pas, et c'est assumé. Colab gratuit suffit : ~1h pour 10M. (Environnement : CPU ou T4, au choix — le temps est le même.)

**3 pas** : 1. télécharge le fichier lourd (vérifié par sha) → 2. éduque 1h (une seule commande) → 3. bilan parlé (optionnel).

In [ ]:
!test -d RATISS-ONE || git clone https://github.com/jonathansearch/RATISS-ONE.git
%cd RATISS-ONE
!python3 --version && git log --oneline -1

## 1. Le fichier lourd (5M, téléchargé + vérifié)
Le labo a pesé le fichier (270 Mo → 35,7 Mo) : téléchargement direct depuis GitHub, vérifié par sha avant de brûler.

In [ ]:
import os
GZ = "/tmp/masse-5M.jsonl.gz"
JSONL = "/tmp/masse-5M.jsonl"
SHA = "b68585abcff40eb37ee86bdff463a50c47a5e47598ef36e4d49ff6a702fac7e5"
URL = "https://raw.githubusercontent.com/jonathansearch/RATISS-ONE/main/education-massive/masse-5M.jsonl.gz"
if not os.path.exists(GZ):
    print("⬇️ téléchargement du fichier lourd (35,7 Mo)…")
    !wget -q -O {GZ} {URL}
import hashlib
sha = hashlib.sha256(open(GZ, "rb").read()).hexdigest()
assert sha == SHA, f"FICHIER CORROMPU : {sha[:16]}"
print("✅ sha OK — fichier intact")
!gzip -dk -f {GZ}
!wc -l {JSONL}

## 2. Éducation 1h (UNE seule commande, ~10M)
Le fichier (5M) puis 5M fraîches (seed 8), même tissu, une nuit par lot de 200k. Laisse tourner ~55 min : chaque lot imprime liens/neurones/écoutes/RAM/temps.

In [ ]:
!python3 education-massive/eduquer_masse.py --entree /tmp/masse-5M.jsonl --lot 200000 --complement 5000000 --bilan /tmp/bilan.txt 2>&1 | grep -v nettoyage | tail -60

## 3. Bilan parlé (optionnel, ~2 min)
Le tissu dit son bilan avec sa vraie bouche (voix FR siwis).

In [ ]:
!pip install -q piper-tts
!python3 -m piper.download_voices fr_FR-siwis-medium --download-dir /tmp/voix 2>&1 | tail -1
phrase = open("/tmp/bilan.txt", encoding="utf-8").read()
print("dit :", phrase)
!python3 -m piper --model /tmp/voix/fr_FR-siwis-medium.onnx --output_file /tmp/bilan.wav < /tmp/bilan.txt
from IPython.display import Audio
Audio("/tmp/bilan.wav")